# Phase 3-2: Distributional Analysis

Analyze how adversarial attacks shift the feature distribution.

In [ ]:
import sys
import os
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.spatial.distance import jensenshannon
from scipy.stats import wasserstein_distance

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

print(f"Colab: {IN_COLAB}")

In [ ]:
# paths
if IN_COLAB:
    BASE = "/content/drive/MyDrive/Colab Notebooks/data"
else:
    BASE = "/Users/tyreecruse/Desktop/CS230/Project/Data"

FEATURES_DIR = f"{BASE}/analysis/yolo_features"
RESULTS_DIR = f"{BASE}/analysis/results"

os.makedirs(RESULTS_DIR, exist_ok=True)

ATTACKS = [
    "fgsm_030", "fgsm_045", "fgsm_060", "fgsm_075", "fgsm_090", "fgsm_105",
    "gaussian_010", "gaussian_050", "gaussian_150", "gaussian_200", "gaussian_250",
    "patches",
]

print(f"Features: {FEATURES_DIR}")
print(f"Results: {RESULTS_DIR}")

In [ ]:
# load clean features
print("Loading clean features...")
with open(f"{FEATURES_DIR}/clean_yolo_features.pkl", 'rb') as f:
    cleanData = pickle.load(f)
cleanFeats = cleanData['features']
print(f"Clean shape: {cleanFeats.shape}")

# compute clean centroid
centroid = cleanFeats.mean(axis=0)
centroid = centroid / np.linalg.norm(centroid)  # normalize

In [ ]:
# analyze each attack
results = []

# compute clean distances first
cleanDist = 1 - (cleanFeats @ centroid)  # cosine distance

for attackName in ATTACKS:
    print(f"\nAnalyzing {attackName}...", end=" ")
    
    try:
        # load adversarial features
        with open(f"{FEATURES_DIR}/{attackName}_yolo_features.pkl", 'rb') as f:
            advData = pickle.load(f)
        advFeats = advData['features']
        
        # compute distances from centroid
        advDist = 1 - (advFeats @ centroid)
        
        # basic stats
        cleanMean = cleanDist.mean()
        cleanStd = cleanDist.std()
        advMean = advDist.mean()
        advStd = advDist.std()
        
        # Cohen's d
        pooledStd = np.sqrt(((len(cleanDist)-1)*cleanStd**2 + (len(advDist)-1)*advStd**2) / 
                           (len(cleanDist) + len(advDist) - 2))
        cohensD = (advMean - cleanMean) / pooledStd
        
        # effect size interpretation
        d = abs(cohensD)
        if d < 0.2:
            effect = 'negligible'
        elif d < 0.5:
            effect = 'small'
        elif d < 0.8:
            effect = 'medium'
        elif d < 1.2:
            effect = 'large'
        else:
            effect = 'very large'
        
        # KS test
        ksStat, ksPval = stats.ks_2samp(cleanDist, advDist)
        
        # Wasserstein distance
        wDist = wasserstein_distance(cleanDist, advDist)
        
        # Jensen-Shannon divergence
        allData = np.concatenate([cleanDist, advDist])
        bins = np.linspace(allData.min(), allData.max(), 51)
        cleanHist, _ = np.histogram(cleanDist, bins=bins, density=True)
        advHist, _ = np.histogram(advDist, bins=bins, density=True)
        cleanHist = cleanHist + 1e-10
        advHist = advHist + 1e-10
        cleanHist = cleanHist / cleanHist.sum()
        advHist = advHist / advHist.sum()
        jsDist = jensenshannon(cleanHist, advHist)
        
        # overlap
        overlap = np.minimum(cleanHist, advHist).sum()
        
        # parse attack type
        if attackName.startswith('fgsm'):
            attackType = 'FGSM'
            strength = int(attackName.split('_')[1]) / 1000
        elif attackName.startswith('gaussian'):
            attackType = 'Gaussian'
            strength = int(attackName.split('_')[1]) / 1000
        else:
            attackType = 'Patch'
            strength = None
        
        results.append({
            'attack': attackName,
            'attackType': attackType,
            'strength': strength,
            'cohensD': cohensD,
            'effect': effect,
            'ksStat': ksStat,
            'ksPval': ksPval,
            'wasserstein': wDist,
            'jensenShannon': jsDist,
            'overlap': overlap,
            'meanShift': advMean - cleanMean,
        })
        
        print(f"d={cohensD:.2f} ({effect})")
        
    except FileNotFoundError:
        print("not found")
    except Exception as e:
        print(f"error: {e}")

print(f"\nAnalyzed {len(results)} attacks")

In [ ]:
# display results
df = pd.DataFrame(results)

print("\n" + "="*70)
print("DISTRIBUTIONAL ANALYSIS")
print("="*70)
print(df[['attackType', 'strength', 'cohensD', 'effect', 'ksStat', 'wasserstein', 'overlap']].round(3).to_string(index=False))

# save
df.to_csv(f"{RESULTS_DIR}/distributional_analysis.csv", index=False)
print(f"\nSaved to {RESULTS_DIR}/distributional_analysis.csv")

In [ ]:
# plot effect size vs strength
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# FGSM
fgsm = df[df['attackType'] == 'FGSM'].sort_values('strength')
axes[0].plot(fgsm['strength'], fgsm['cohensD'], 'ro-', markersize=10, linewidth=2)
axes[0].axhline(0.8, color='orange', linestyle='--', label='Large (0.8)')
axes[0].axhline(0.5, color='gray', linestyle='--', label='Medium (0.5)')
axes[0].set_xlabel('FGSM ε')
axes[0].set_ylabel("Cohen's d")
axes[0].set_title('FGSM: Effect Size')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Gaussian
gauss = df[df['attackType'] == 'Gaussian'].sort_values('strength')
axes[1].plot(gauss['strength'], gauss['cohensD'], 'go-', markersize=10, linewidth=2)
axes[1].axhline(0.8, color='orange', linestyle='--', label='Large (0.8)')
axes[1].axhline(0.5, color='gray', linestyle='--', label='Medium (0.5)')
axes[1].set_xlabel('Gaussian σ')
axes[1].set_ylabel("Cohen's d")
axes[1].set_title('Gaussian: Effect Size')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/effect_size_by_strength.png", dpi=150)
plt.show()

In [ ]:
# plot distance distributions for a few attacks
toPlot = ['fgsm_045', 'fgsm_075', 'fgsm_105', 'gaussian_050', 'gaussian_250', 'patches']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, name in enumerate(toPlot):
    ax = axes[i]
    
    try:
        with open(f"{FEATURES_DIR}/{name}_yolo_features.pkl", 'rb') as f:
            advData = pickle.load(f)
        advFeats = advData['features']
        advDist = 1 - (advFeats @ centroid)
        
        ax.hist(cleanDist, bins=40, alpha=0.6, density=True, label='Clean', color='blue')
        ax.hist(advDist, bins=40, alpha=0.6, density=True, label=name, color='red')
        
        # 3-sigma threshold
        thresh = cleanDist.mean() + 3 * cleanDist.std()
        ax.axvline(thresh, color='green', linestyle='--', linewidth=2, label='3σ')
        
        ax.set_xlabel('Distance from Centroid')
        ax.set_ylabel('Density')
        ax.set_title(name)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    except:
        ax.set_title(f"{name} (not found)")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/distance_distributions.png", dpi=150)
plt.show()

In [ ]:
print("\n" + "="*50)
print("KEY FINDINGS")
print("="*50)

# FGSM
print("\nFGSM:")
for _, r in fgsm.iterrows():
    status = "Detectable" if r['cohensD'] > 0.8 else "Marginal" if r['cohensD'] > 0.5 else "Hard"
    print(f"  ε={r['strength']:.3f}: d={r['cohensD']:.2f} ({r['effect']}) - {status}")

# Gaussian
print("\nGaussian:")
for _, r in gauss.iterrows():
    status = "Detectable" if r['cohensD'] > 0.8 else "Marginal" if r['cohensD'] > 0.5 else "Hard"
    print(f"  σ={r['strength']:.3f}: d={r['cohensD']:.2f} ({r['effect']}) - {status}")

# Patches
patch = df[df['attackType'] == 'Patch']
if len(patch) > 0:
    r = patch.iloc[0]
    print(f"\nPatches: d={r['cohensD']:.2f} ({r['effect']})")